# Import

In [1]:
!pip install googletrans==4.0.0-rc1  

In [2]:
import en_core_web_sm
import json
import numpy as np
import random
import re
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
)
from typing import Any, List, Mapping, Tuple

In [3]:
from transformers import BertTokenizer, BertModel
from bert_score import BERTScorer

scorer = BERTScorer(model_type='bert-base-multilingual-cased')

# TEXT

In [4]:
from googletrans import Translator

# Define input context text
text_context = "Monas atau Monumen Nasional adalah landmark ikonik yang terletak di Jakarta, Indonesia. Monumen ini dibangun untuk memperingati perjuangan kemerdekaan Indonesia dan diresmikan pada tahun 1975. Monas memiliki tinggi sekitar 132 meter dan bagian puncaknya dilapisi emas seberat 50 kilogram. Di dalam Monas, terdapat museum sejarah yang berisi diorama perjalanan perjuangan bangsa Indonesia. Pengunjung juga dapat naik ke puncak monumen menggunakan lift untuk menikmati pemandangan kota Jakarta dari ketinggian. Monas menjadi salah satu destinasi wisata utama di ibu kota dan sering digunakan sebagai tempat upacara kenegaraan serta berbagai acara besar lainnya. Selain itu, kawasan sekitar Monas juga merupakan ruang terbuka hijau yang sering dimanfaatkan masyarakat untuk olahraga dan rekreasi. Monas tidak hanya menjadi simbol perjuangan nasional, tetapi juga bagian penting dari identitas budaya Indonesia."
# Create translator object
translator = Translator()

# Translate the text
input_text = translator.translate(text_context, src="id", dest="en").text

# Print the translated text
print("Translated Text:")
print(input_text)

Translated Text:
Monas or National Monument is an iconic landmark located in Jakarta, Indonesia.This monument was built to commemorate the struggle for Indonesian independence and was inaugurated in 1975. Monas has a height of around 132 meters and the peak part is coated with 50 kilograms of gold.In Monas, there is a historical museum containing the diorama of the journey of the struggle of the Indonesian people.Visitors can also go up to the top of the monument using an elevator to enjoy the view of the city of Jakarta from a height.Monas is one of the main tourist destinations in the Capital City and is often used as a place for state ceremonies and various other major events.In addition, the area around Monas is also a green open space that is often used by the community for sports and recreation.Monas is not only a symbol of national struggle, but also an important part of Indonesian cultural identity.


# QA GENERATOR

In [5]:
class QuestionGenerator:
    def __init__(self, model_name: str) -> None:
        self.ANSWER_TOKEN = "<answer>"
        self.CONTEXT_TOKEN = "<context>"
        self.SEQ_LENGTH = 512
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Load model dan tokenizer berdasarkan parameter yang diberikan
        self.qg_tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        self.qg_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.qg_model.to(self.device)
        self.qg_model.eval()

        self.qa_evaluator = QAEvaluator()

    def generate(
        self,
        article: str,
        use_evaluator: bool = True,
        num_questions: bool = None,
        answer_style: str = "all"
    ) -> List:
        print("Generating questions...\n")

        qg_inputs, qg_answers = self.generate_qg_inputs(article, answer_style)
        generated_questions = self.generate_questions_from_inputs(qg_inputs)

        message = "{} questions doesn't match {} answers".format(
            len(generated_questions), len(qg_answers)
        )
        assert len(generated_questions) == len(qg_answers), message

        if use_evaluator:
            print("Evaluating QA pairs...\n")
            encoded_qa_pairs = self.qa_evaluator.encode_qa_pairs(
                generated_questions, qg_answers
            )
            scores = self.qa_evaluator.get_scores(encoded_qa_pairs)

            if num_questions:
                qa_list = self._get_ranked_qa_pairs(
                    generated_questions, qg_answers, scores, num_questions
                )
            else:
                qa_list = self._get_ranked_qa_pairs(
                    generated_questions, qg_answers, scores
                )

        else:
            print("Skipping evaluation step.\n")
            qa_list = self._get_all_qa_pairs(generated_questions, qg_answers)

        return qa_list

    def generate_qg_inputs(self, text: str, answer_style: str) -> Tuple[List[str], List[str]]:
        VALID_ANSWER_STYLES = ["all", "sentences", "multiple_choice"]

        if answer_style not in VALID_ANSWER_STYLES:
            raise ValueError(
                "Invalid answer style {}. Please choose from {}".format(
                    answer_style, VALID_ANSWER_STYLES
                )
            )

        inputs = []
        answers = []

        if answer_style == "sentences" or answer_style == "all":
            segments = self._split_into_segments(text)

            for segment in segments:
                sentences = self._split_text(segment)
                prepped_inputs, prepped_answers = self._prepare_qg_inputs(
                    sentences, segment
                )
                inputs.extend(prepped_inputs)
                answers.extend(prepped_answers)

        if answer_style == "multiple_choice" or answer_style == "all":
            sentences = self._split_text(text)
            prepped_inputs, prepped_answers = self._prepare_qg_inputs_MC(
                sentences
            )
            inputs.extend(prepped_inputs)
            answers.extend(prepped_answers)

        return inputs, answers

    def generate_questions_from_inputs(self, qg_inputs: List) -> List[str]:
        generated_questions = []

        for qg_input in qg_inputs:
            question = self._generate_question(qg_input)
            generated_questions.append(question)

        return generated_questions

    def _split_text(self, text: str) -> List[str]:
        MAX_SENTENCE_LEN = 128
        sentences = re.findall(".*?[.!\?]", text)
        cut_sentences = []

        for sentence in sentences:
            if len(sentence) > MAX_SENTENCE_LEN:
                cut_sentences.extend(re.split("[,;:)]", sentence))

        # remove useless post-quote sentence fragments
        cut_sentences = [s for s in sentences if len(s.split(" ")) > 5]
        sentences = sentences + cut_sentences

        return list(set([s.strip(" ") for s in sentences]))

    def _split_into_segments(self, text: str) -> List[str]:
        MAX_TOKENS = 490
        paragraphs = text.split("\n")
        tokenized_paragraphs = [
            self.qg_tokenizer(p)["input_ids"] for p in paragraphs if len(p) > 0
        ]
        segments = []

        while len(tokenized_paragraphs) > 0:
            segment = []

            while len(segment) < MAX_TOKENS and len(tokenized_paragraphs) > 0:
                paragraph = tokenized_paragraphs.pop(0)
                segment.extend(paragraph)
            segments.append(segment)

        return [self.qg_tokenizer.decode(s, skip_special_tokens=True) for s in segments]

    def _prepare_qg_inputs(
        self,
        sentences: List[str],
        text: str
    ) -> Tuple[List[str], List[str]]:
        inputs = []
        answers = []

        for sentence in sentences:
            qg_input = f"{self.ANSWER_TOKEN} {sentence} {self.CONTEXT_TOKEN} {text}"
            inputs.append(qg_input)
            answers.append(sentence)

        return inputs, answers

    def _prepare_qg_inputs_MC(self, sentences: List[str]) -> Tuple[List[str], List[str]]:
        spacy_nlp = en_core_web_sm.load()
        docs = list(spacy_nlp.pipe(sentences, disable=["parser"]))
        inputs_from_text = []
        answers_from_text = []

        for doc, sentence in zip(docs, sentences):
            entities = doc.ents
            if entities:

                for entity in entities:
                    qg_input = f"{self.ANSWER_TOKEN} {entity} {self.CONTEXT_TOKEN} {sentence}"
                    answers = self._get_MC_answers(entity, docs)
                    inputs_from_text.append(qg_input)
                    answers_from_text.append(answers)

        return inputs_from_text, answers_from_text

    def _get_MC_answers(self, correct_answer: Any, docs: Any) -> List[Mapping[str, Any]]:
        entities = []

        for doc in docs:
            entities.extend([{"text": e.text, "label_": e.label_}
                            for e in doc.ents])

        entities_json = [json.dumps(kv) for kv in entities]
        pool = set(entities_json)
        num_choices = (
            min(4, len(pool)) - 1
        )

        final_choices = []
        correct_label = correct_answer.label_
        final_choices.append({"answer": correct_answer.text, "correct": True})
        pool.remove(
            json.dumps({"text": correct_answer.text,
                       "label_": correct_answer.label_})
        )

        matches = [e for e in pool if correct_label in e]

        if len(matches) < num_choices:
            choices = matches
            pool = pool.difference(set(choices))
            choices.extend(random.sample(pool, num_choices - len(choices)))
        else:
            choices = random.sample(matches, num_choices)

        choices = [json.loads(s) for s in choices]

        for choice in choices:
            final_choices.append({"answer": choice["text"], "correct": False})

        random.shuffle(final_choices)
        return final_choices

    @torch.no_grad()
    def _generate_question(self, qg_input: str) -> str:
        encoded_input = self._encode_qg_input(qg_input)
        output = self.qg_model.generate(input_ids=encoded_input["input_ids"])
        question = self.qg_tokenizer.decode(
            output[0],
            skip_special_tokens=True
        )
        return question

    def _encode_qg_input(self, qg_input: str) -> torch.tensor:
        return self.qg_tokenizer(
            qg_input,
            padding='max_length',
            max_length=self.SEQ_LENGTH,
            truncation=True,
            return_tensors="pt",
        ).to(self.device)

    def _get_ranked_qa_pairs(
        self, generated_questions: List[str], qg_answers: List[str], scores, num_questions: int = 10
    ) -> List[Mapping[str, str]]:
        if num_questions > len(scores):
            num_questions = len(scores)
            print((
                f"\nWas only able to generate {num_questions} questions.",
                "For more questions, please input a longer text.")
            )

        qa_list = []

        for i in range(num_questions):
            index = scores[i]
            qa = {
                "question": generated_questions[index].split("?")[0] + "?",
                "answer": qg_answers[index]
            }
            qa_list.append(qa)

        return qa_list

    def _get_all_qa_pairs(self, generated_questions: List[str], qg_answers: List[str]):
        qa_list = []

        for question, answer in zip(generated_questions, qg_answers):
            qa = {
                "question": question.split("?")[0] + "?",
                "answer": answer
            }
            qa_list.append(qa)

        return qa_list


class QAEvaluator:
    def __init__(self) -> None:

        QAE_PRETRAINED = "iarfmoose/bert-base-cased-qa-evaluator"
        self.SEQ_LENGTH = 512

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu")

        self.qae_tokenizer = AutoTokenizer.from_pretrained(QAE_PRETRAINED)
        self.qae_model = AutoModelForSequenceClassification.from_pretrained(
            QAE_PRETRAINED
        )
        self.qae_model.to(self.device)
        self.qae_model.eval()

    def encode_qa_pairs(self, questions: List[str], answers: List[str]) -> List[torch.tensor]:
        encoded_pairs = []

        for question, answer in zip(questions, answers):
            encoded_qa = self._encode_qa(question, answer)
            encoded_pairs.append(encoded_qa.to(self.device))

        return encoded_pairs

    def get_scores(self, encoded_qa_pairs: List[torch.tensor]) -> List[float]:
        scores = {}

        for i in range(len(encoded_qa_pairs)):
            scores[i] = self._evaluate_qa(encoded_qa_pairs[i])

        return [
            k for k, v in sorted(scores.items(), key=lambda item: item[1], reverse=True)
        ]

    def _encode_qa(self, question: str, answer: str) -> torch.tensor:
        if type(answer) is list:
            for a in answer:
                if a["correct"]:
                    correct_answer = a["answer"]
        else:
            correct_answer = answer

        return self.qae_tokenizer(
            text=question,
            text_pair=correct_answer,
            padding="max_length",
            max_length=self.SEQ_LENGTH,
            truncation=True,
            return_tensors="pt",
        )

    @torch.no_grad()
    def _evaluate_qa(self, encoded_qa_pair: torch.tensor) -> float:
        output = self.qae_model(**encoded_qa_pair)
        return output[0][0][1]


def print_qa(qa_list: List[Mapping[str, str]], show_answers: bool = True) -> None:
    for i in range(len(qa_list)):

        space = " " * int(np.where(i < 9, 3, 4))

        print(f"{i + 1}) Q: {qa_list[i]['question']}")

        answer = qa_list[i]["answer"]

        if type(answer) is list:

            if show_answers:
                print(
                    f"{space}A: 1. {answer[0]['answer']} "
                    f"{np.where(answer[0]['correct'], '(correct)', '')}"
                )
                for j in range(1, len(answer)):
                    print(
                        f"{space + '   '}{j + 1}. {answer[j]['answer']} "
                        f"{np.where(answer[j]['correct']==True,'(correct)', '')}"
                    )

            else:
                print(f"{space}A: 1. {answer[0]['answer']}")
                for j in range(1, len(answer)):
                    print(f"{space + '   '}{j + 1}. {answer[j]['answer']}")

            print("")

        else:
            if show_answers:
                print(f"{space}A: {answer}\n")

## BERT

In [6]:
tokenizer_bert = AutoTokenizer.from_pretrained("mrm8488/bert2bert-multilingual_shared-question-generation")
model_bert = AutoModelForSeq2SeqLM.from_pretrained("mrm8488/bert2bert-multilingual_shared-question-generation")

Config of the encoder: <class 'transformers.models.bert.modeling_bert.BertModel'> is overwritten by shared encoder config: BertConfig {
  "_name_or_path": "bert-base-multilingual-cased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "directionality": "bidi",
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler_type": "first_token_transform",
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.49.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 119547
}

In [7]:
encoded_input_bert = tokenizer_bert(input_text, return_tensors="pt")

In [8]:
labels_bert = encoded_input_bert['input_ids'].clone()

In [9]:
def mask_label_padding_bert(labels):
    MASK_ID = -100
    labels_bert[labels_bert==tokenizer_bert.pad_token_id] = MASK_ID
    return labels_bert
masked_labels_bert = mask_label_padding_bert(labels_bert)

In [10]:
outputs_bert = model_bert(
    input_ids=encoded_input_bert['input_ids'],
    attention_mask=encoded_input_bert['attention_mask'],
    labels=masked_labels_bert # Use masked labels specific to BERT
)

c:\Users\USER\anaconda3\lib\site-packages\transformers\models\encoder_decoder\modeling_encoder_decoder.py:629: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than tensor.new_tensor(sourceTensor).
  decoder_attention_mask = decoder_input_ids.new_tensor(decoder_input_ids != self.config.pad_token_id)
c:\Users\USER\anaconda3\lib\site-packages\transformers\models\encoder_decoder\modeling_encoder_decoder.py:649: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)


In [11]:
loss_bert, prediction_scores = outputs_bert[:2]

In [12]:
loss_bert = outputs_bert.loss

In [13]:
print(f"Loss: {loss_bert.item()}")

Loss: 9.225340843200684


In [14]:
qg_bert2bert = QuestionGenerator("mrm8488/bert2bert-multilingual_shared-question-generation")

Config of the encoder: <class 'transformers.models.bert.modeling_bert.BertModel'> is overwritten by shared encoder config: BertConfig {
  "_name_or_path": "bert-base-multilingual-cased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "directionality": "bidi",
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler_type": "first_token_transform",
  "position_embedding_type": "absolute",
  "torch_dtype": "float32",
  "transformers_version": "4.49.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 119547
}

In [15]:
qa_list_bert = qg_bert2bert.generate(
    input_text,
    num_questions=10,  # Generate 10 questions
    answer_style='sentences'  # Restrict answers to short-form only
)

Generating questions...

Evaluating QA pairs...

('\nWas only able to generate 8 questions.', 'For more questions, please input a longer text.')


In [16]:
print("BERT")
print_qa(qa_list_bert)

BERT
1) Q: What is the national symbol of Monas?
   A: Monas or National Monument is an iconic landmark located in Jakarta, Indonesia.

2) Q: What does the Monas symbolize independence?
   A: Monas is not only a symbol of national struggle, but also an important part of Indonesian cultural identity.

3) Q: What does Monas symbolize?
   A: In addition, the area around Monas is also a green open space that is often used by the community for sports and recreation.

4) Q: What is the origin of the name Monas?
   A: Monas is one of the main tourist destinations in the Capital City and is often used as a place for state ceremonies and various other major events.

5) Q: How tall is Monas?
   A: Monas has a height of around 132 meters and the peak part is coated with 50 kilograms of gold.

6) Q: What is the national symbol of Monas?
   A: In Monas, there is a historical museum containing the diorama of the journey of the struggle of the Indonesian people.

7) Q: What does Monas symbolize?
   A

In [17]:
reference = input_text  # Menggunakan teks asli sebagai referensi

# Menghitung BERTScore untuk setiap pasangan QA
print("BERT - BERTScore per QA Pair:\n")

for i, qa in enumerate(qa_list_bert):
    candidate = qa['question'] + " " + qa['answer']
    P, R, F1 = scorer.score([candidate], [reference])

    print(f"QA {i+1}:")
    print(f"  Precision: {P.mean():.4f}")
    print(f"  Recall: {R.mean():.4f}")
    print(f"  F1-score: {F1.mean():.4f}\n")


BERT - BERTScore per QA Pair:

QA 1:
  Precision: 0.8196
  Recall: 0.6473
  F1-score: 0.7233

QA 2:
  Precision: 0.8534
  Recall: 0.6589
  F1-score: 0.7437

QA 3:
  Precision: 0.8615
  Recall: 0.6812
  F1-score: 0.7608

QA 4:
  Precision: 0.8680
  Recall: 0.6852
  F1-score: 0.7659

QA 5:
  Precision: 0.8220
  Recall: 0.6417
  F1-score: 0.7208

QA 6:
  Precision: 0.8432
  Recall: 0.6782
  F1-score: 0.7518

QA 7:
  Precision: 0.8335
  Recall: 0.6742
  F1-score: 0.7454

QA 8:
  Precision: 0.8222
  Recall: 0.6484
  F1-score: 0.7250



## T5

In [18]:
tokenizer_t5 = AutoTokenizer.from_pretrained("potsawee/t5-large-generation-squad-QuestionAnswer")
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("potsawee/t5-large-generation-squad-QuestionAnswer")

In [19]:
# Tokenize input text
encoded_input_t5 = tokenizer_t5(input_text, return_tensors="pt")

In [20]:
# Prepare labels for training
labels_t5 = encoded_input_t5['input_ids'].clone()

In [21]:
# Mask padding tokens
def mask_label_padding_t5(labels_t5):
    MASK_ID = -100
    labels_t5[labels_t5==tokenizer_t5.pad_token_id] = MASK_ID
    return labels_t5
masked_labels_t5 = mask_label_padding_t5(labels_t5)

In [22]:
# Forward pass through T5 model
outputs_t5 = model_t5(
    input_ids=encoded_input_t5["input_ids"],
    attention_mask=encoded_input_t5["attention_mask"],
    labels=labels_t5
)

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


In [23]:
loss_t5, prediction_scores = outputs_t5[:2]

In [24]:
loss_t5 = outputs_t5.loss

In [25]:
print(f"Loss: {loss_t5.item()}")

Loss: 7.707540988922119


In [26]:
qg_t5 = QuestionGenerator("potsawee/t5-large-generation-squad-QuestionAnswer")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [27]:
# Generate **Short Answer** Questions based on `input_text`
qa_list_t5 = qg_t5.generate(
    input_text,
    num_questions=10,  # Generate 10 questions
    answer_style='sentences'  # Restrict answers to short-form only
)


Generating questions...

Evaluating QA pairs...

('\nWas only able to generate 8 questions.', 'For more questions, please input a longer text.')


In [28]:
# Print the generated QA pairs
print("T5")
print_qa(qa_list_t5)

T5
1) Q: What is the name of the iconic landmark in Jakarta?
   A: Monas is not only a symbol of national struggle, but also an important part of Indonesian cultural identity.

2) Q: What is the name of the monument that is located in Jakarta?
   A: This monument was built to commemorate the struggle for Indonesian independence and was inaugurated in 1975.

3) Q: What is the name of the monument that is located in Jakarta?
   A: Visitors can also go up to the top of the monument using an elevator to enjoy the view of the city of Jakarta from a height.

4) Q: What is the height of the Monas?
   A: Monas or National Monument is an iconic landmark located in Jakarta, Indonesia.

5) Q: What is the height of the Monas?
   A: In Monas, there is a historical museum containing the diorama of the journey of the struggle of the Indonesian people.

6) Q: What is the name of the monument that was built to commemorate the struggle for Indonesian independence?
   A: Monas is one of the main tourist 

In [29]:
reference = input_text  # Menggunakan teks asli sebagai referensi

# Menghitung BERTScore untuk setiap pasangan QA
print("T5 - BERTScore per QA Pair:\n")

for i, qa in enumerate(qa_list_t5):
    candidate = qa['question'] + " " + qa['answer']
    P, R, F1 = scorer.score([candidate], [reference])

    print(f"QA {i+1}:")
    print(f"  Precision: {P.mean():.4f}")
    print(f"  Recall: {R.mean():.4f}")
    print(f"  F1-score: {F1.mean():.4f}\n")


T5 - BERTScore per QA Pair:

QA 1:
  Precision: 0.8582
  Recall: 0.6831
  F1-score: 0.7607

QA 2:
  Precision: 0.8154
  Recall: 0.6596
  F1-score: 0.7292

QA 3:
  Precision: 0.8394
  Recall: 0.6842
  F1-score: 0.7539

QA 4:
  Precision: 0.8166
  Recall: 0.6468
  F1-score: 0.7218

QA 5:
  Precision: 0.8317
  Recall: 0.6737
  F1-score: 0.7444

QA 6:
  Precision: 0.8729
  Recall: 0.7285
  F1-score: 0.7942

QA 7:
  Precision: 0.8502
  Recall: 0.6417
  F1-score: 0.7314

QA 8:
  Precision: 0.8435
  Recall: 0.6763
  F1-score: 0.7507



## Pegasus X

In [30]:
tokenizer_pegasusx = AutoTokenizer.from_pretrained("google/pegasus-xsum")
model_pegasusx = AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-xsum")

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
encoded_input_pegasusx = tokenizer_pegasusx(input_text, return_tensors="pt")

In [32]:
labels_pegasusx = encoded_input_pegasusx['input_ids'].clone()

In [33]:
def mask_label_padding_pegasusx(labels_pegasusx):
    MASK_ID = -100
    labels_pegasusx[labels_pegasusx==tokenizer_pegasusx.pad_token_id] = MASK_ID
    return labels_pegasusx
masked_labels_pegasusx = mask_label_padding_pegasusx(labels_pegasusx)

In [34]:
outputs_pegasusx = model_pegasusx(
    input_ids=encoded_input_pegasusx['input_ids'],
    attention_mask=encoded_input_pegasusx['attention_mask'],
    labels=masked_labels_pegasusx # Use masked labels specific to PegasusX
)


In [35]:
loss_pegasusx, prediction_scores = outputs_pegasusx[:2]

In [36]:
loss_pegasusx = outputs_pegasusx.loss

In [37]:
print(f"Loss: {loss_pegasusx.item()}")

Loss: 0.4426846206188202


In [38]:
qg_pegasusx = QuestionGenerator("google/pegasus-xsum")

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [39]:
qa_list_pegasusx = qg_pegasusx.generate(
    input_text,
    num_questions=10,  # Generate 10 questions
    answer_style='sentences'  # Restrict answers to short-form only
)



Generating questions...

Evaluating QA pairs...

('\nWas only able to generate 8 questions.', 'For more questions, please input a longer text.')


In [40]:
print("PEGASUS-X")
print_qa(qa_list_pegasusx)

PEGASUS-X
1) Q: How much do you know about the Monas monument in Jakarta, Indonesia?
   A: Monas is one of the main tourist destinations in the Capital City and is often used as a place for state ceremonies and various other major events.

2) Q: How much do you know about Monas or the National Monument of Indonesia?
   A: In Monas, there is a historical museum containing the diorama of the journey of the struggle of the Indonesian people.

3) Q: How much do you know about Monas or the National Monument of Indonesia?
   A: Monas is not only a symbol of national struggle, but also an important part of Indonesian cultural identity.

4) Q: What is Monas or the National Monument of Indonesia?
   A: Monas has a height of around 132 meters and the peak part is coated with 50 kilograms of gold.

5) Q: How much do you know about Monas or the National Monument of Indonesia?
   A: This monument was built to commemorate the struggle for Indonesian independence and was inaugurated in 1975.

6) Q: H

In [41]:
reference = input_text  # Menggunakan teks asli sebagai referensi

# Menghitung BERTScore untuk setiap pasangan QA
print("PegasusX - BERTScore per QA Pair:\n")

for i, qa in enumerate(qa_list_pegasusx):
    candidate = qa['question'] + " " + qa['answer']
    P, R, F1 = scorer.score([candidate], [reference])
 
    print(f"QA {i+1}:")
    print(f"  Precision: {P.mean():.4f}")
    print(f"  Recall: {R.mean():.4f}")
    print(f"  F1-score: {F1.mean():.4f}\n")


PegasusX - BERTScore per QA Pair:

QA 1:
  Precision: 0.8537
  Recall: 0.7028
  F1-score: 0.7709

QA 2:
  Precision: 0.8106
  Recall: 0.6781
  F1-score: 0.7385

QA 3:
  Precision: 0.8265
  Recall: 0.6700
  F1-score: 0.7401

QA 4:
  Precision: 0.8334
  Recall: 0.6692
  F1-score: 0.7423

QA 5:
  Precision: 0.7860
  Recall: 0.6599
  F1-score: 0.7174

QA 6:
  Precision: 0.8332
  Recall: 0.6990
  F1-score: 0.7602

QA 7:
  Precision: 0.7826
  Recall: 0.6453
  F1-score: 0.7074

QA 8:
  Precision: 0.8186
  Recall: 0.6797
  F1-score: 0.7427



# Translate Indonesia

## BERT

In [42]:
from googletrans import Translator

# Buat objek penerjemah
translator = Translator()

# Fungsi untuk menerjemahkan daftar QA yang dihasilkan
def translate_qa_list(qa_list, src_lang="en", dest_lang="id"):
    translated_qa_list = []
    for qa in qa_list:
        # Check if the answer is valid before translating
        if qa["answer"]:
            translated_qa = {
                "question": translator.translate(qa["question"], src=src_lang, dest=dest_lang).text,
                "answer": translator.translate(qa["answer"], src=src_lang, dest=dest_lang).text
            }
            translated_qa_list.append(translated_qa)
        else:
            print(f"Skipping translation for question: {qa['question']} due to empty answer.")
    return translated_qa_list

# Menerjemahkan isi dari qa_list_PegasusX ke Bahasa Indonesia dan menyimpannya
qa_list_BERTindo = translate_qa_list(qa_list_bert, src_lang="en", dest_lang="id")

# Menampilkan hasil terjemahan
print("Translated QA List (Indonesian):")
for qa in qa_list_BERTindo:
    print(f"Q: {qa['question']}")
    print(f"A: {qa['answer']}")
    print()

Translated QA List (Indonesian):
Q: Apa simbol nasional monas?
A: Monas atau Monumen Nasional adalah tengara ikonik yang terletak di Jakarta, Indonesia.

Q: Apa yang melambangkan monas kemerdekaan?
A: Monas tidak hanya merupakan simbol perjuangan nasional, tetapi juga bagian penting dari identitas budaya Indonesia.

Q: Apa yang dilambangkan Monas?
A: Selain itu, area di sekitar Monas juga merupakan ruang terbuka hijau yang sering digunakan oleh komunitas untuk olahraga dan rekreasi.

Q: Apa asal nama Monas?
A: Monas adalah salah satu tujuan wisata utama di ibu kota dan sering digunakan sebagai tempat untuk upacara negara dan berbagai acara besar lainnya.

Q: Berapa tinggi monas?
A: Monas memiliki ketinggian sekitar 132 meter dan bagian puncak dilapisi dengan 50 kilogram emas.

Q: Apa simbol nasional monas?
A: Di Monas, ada museum bersejarah yang berisi diorama perjalanan perjuangan rakyat Indonesia.

Q: Apa yang dilambangkan Monas?
A: Pengunjung juga dapat naik ke puncak monumen menggu

### Bert Score Indo

In [43]:
reference = text_context  # Menggunakan teks asli sebagai referensi

# Menghitung BERTScore untuk setiap pasangan QA
print("BERT - BERTScore per QA Pair:\n")

for i, qa in enumerate(qa_list_BERTindo):
    candidate = qa['question'] + " " + qa['answer']
    P, R, F1 = scorer.score([candidate], [reference])

    print(f"QA {i+1}:")
    print(f"  Precision: {P.mean():.4f}")
    print(f"  Recall: {R.mean():.4f}")
    print(f"  F1-score: {F1.mean():.4f}\n")


BERT - BERTScore per QA Pair:

QA 1:
  Precision: 0.7918
  Recall: 0.6312
  F1-score: 0.7025

QA 2:
  Precision: 0.8377
  Recall: 0.6668
  F1-score: 0.7426

QA 3:
  Precision: 0.8242
  Recall: 0.6745
  F1-score: 0.7419

QA 4:
  Precision: 0.8338
  Recall: 0.6650
  F1-score: 0.7399

QA 5:
  Precision: 0.7758
  Recall: 0.6236
  F1-score: 0.6914

QA 6:
  Precision: 0.7847
  Recall: 0.6575
  F1-score: 0.7155

QA 7:
  Precision: 0.8118
  Recall: 0.6635
  F1-score: 0.7302

QA 8:
  Precision: 0.8116
  Recall: 0.6482
  F1-score: 0.7208



## T5

In [44]:
from googletrans import Translator

# Buat objek penerjemah
translator = Translator()

# Fungsi untuk menerjemahkan daftar QA yang dihasilkan
def translate_qa_list(qa_list, src_lang="en", dest_lang="id"):
    translated_qa_list = []
    for qa in qa_list:
        # Check if the answer is valid before translating
        if qa["answer"]:
            translated_qa = {
                "question": translator.translate(qa["question"], src=src_lang, dest=dest_lang).text,
                "answer": translator.translate(qa["answer"], src=src_lang, dest=dest_lang).text
            }
            translated_qa_list.append(translated_qa)
        else:
            print(f"Skipping translation for question: {qa['question']} due to empty answer.")
    return translated_qa_list

# Menerjemahkan isi dari qa_list_PegasusX ke Bahasa Indonesia dan menyimpannya
qa_list_T5indo = translate_qa_list(qa_list_t5, src_lang="en", dest_lang="id")

# Menampilkan hasil terjemahan
print("Translated QA List (Indonesian):")
for qa in qa_list_T5indo:
    print(f"Q: {qa['question']}")
    print(f"A: {qa['answer']}")
    print()

Translated QA List (Indonesian):
Q: Apa nama landmark ikonik di Jakarta?
A: Monas tidak hanya merupakan simbol perjuangan nasional, tetapi juga bagian penting dari identitas budaya Indonesia.

Q: Apa nama monumen yang terletak di Jakarta?
A: Monumen ini dibangun untuk memperingati perjuangan untuk kemerdekaan Indonesia dan diresmikan pada tahun 1975.

Q: Apa nama monumen yang terletak di Jakarta?
A: Pengunjung juga dapat naik ke puncak monumen menggunakan lift untuk menikmati pemandangan kota Jakarta dari ketinggian.

Q: Berapa ketinggian monas?
A: Monas atau Monumen Nasional adalah tengara ikonik yang terletak di Jakarta, Indonesia.

Q: Berapa ketinggian monas?
A: Di Monas, ada museum bersejarah yang berisi diorama perjalanan perjuangan rakyat Indonesia.

Q: Apa nama monumen yang dibangun untuk memperingati perjuangan untuk kemerdekaan Indonesia?
A: Monas adalah salah satu tujuan wisata utama di ibu kota dan sering digunakan sebagai tempat untuk upacara negara dan berbagai acara besar

### BERT Score Indo

In [45]:
reference = text_context  # Menggunakan teks asli sebagai referensi

# Menghitung BERTScore untuk setiap pasangan QA
print("T5 - BERTScore per QA Pair:\n")

for i, qa in enumerate(qa_list_T5indo):
    candidate = qa['question'] + " " + qa['answer']
    P, R, F1 = scorer.score([candidate], [reference])

    print(f"QA {i+1}:")
    print(f"  Precision: {P.mean():.4f}")
    print(f"  Recall: {R.mean():.4f}")
    print(f"  F1-score: {F1.mean():.4f}\n")


T5 - BERTScore per QA Pair:

QA 1:
  Precision: 0.8757
  Recall: 0.6769
  F1-score: 0.7636

QA 2:
  Precision: 0.8235
  Recall: 0.6581
  F1-score: 0.7315

QA 3:
  Precision: 0.8255
  Recall: 0.6681
  F1-score: 0.7385

QA 4:
  Precision: 0.7720
  Recall: 0.6237
  F1-score: 0.6900

QA 5:
  Precision: 0.7732
  Recall: 0.6558
  F1-score: 0.7097

QA 6:
  Precision: 0.8512
  Recall: 0.7156
  F1-score: 0.7775

QA 7:
  Precision: 0.8016
  Recall: 0.6160
  F1-score: 0.6966

QA 8:
  Precision: 0.8075
  Recall: 0.6812
  F1-score: 0.7390



## Pegasus X

In [46]:
from googletrans import Translator

# Buat objek penerjemah
translator = Translator()

# Fungsi untuk menerjemahkan daftar QA yang dihasilkan
def translate_qa_list(qa_list, src_lang="en", dest_lang="id"):
    translated_qa_list = []
    for qa in qa_list:
        # Check if the answer is valid before translating
        if qa["answer"]:
            translated_qa = {
                "question": translator.translate(qa["question"], src=src_lang, dest=dest_lang).text,
                "answer": translator.translate(qa["answer"], src=src_lang, dest=dest_lang).text
            }
            translated_qa_list.append(translated_qa)
        else:
            print(f"Skipping translation for question: {qa['question']} due to empty answer.")
    return translated_qa_list

# Menerjemahkan isi dari qa_list_PegasusX ke Bahasa Indonesia dan menyimpannya
qa_list_PegasusXindo = translate_qa_list(qa_list_pegasusx, src_lang="en", dest_lang="id")

# Menampilkan hasil terjemahan
print("Translated QA List (Indonesian):")
for qa in qa_list_PegasusXindo:
    print(f"Q: {qa['question']}")
    print(f"A: {qa['answer']}")
    print()

Translated QA List (Indonesian):
Q: Berapa banyak yang Anda ketahui tentang monumen Monas di Jakarta, Indonesia?
A: Monas adalah salah satu tujuan wisata utama di ibu kota dan sering digunakan sebagai tempat untuk upacara negara dan berbagai acara besar lainnya.

Q: Berapa banyak yang Anda ketahui tentang Monas atau Monumen Nasional Indonesia?
A: Di Monas, ada museum bersejarah yang berisi diorama perjalanan perjuangan rakyat Indonesia.

Q: Berapa banyak yang Anda ketahui tentang Monas atau Monumen Nasional Indonesia?
A: Monas tidak hanya merupakan simbol perjuangan nasional, tetapi juga bagian penting dari identitas budaya Indonesia.

Q: Apa itu Monas atau Monumen Nasional Indonesia?
A: Monas memiliki ketinggian sekitar 132 meter dan bagian puncak dilapisi dengan 50 kilogram emas.

Q: Berapa banyak yang Anda ketahui tentang Monas atau Monumen Nasional Indonesia?
A: Monumen ini dibangun untuk memperingati perjuangan untuk kemerdekaan Indonesia dan diresmikan pada tahun 1975.

Q: Berapa

### BERT Score Indo

In [47]:
reference = text_context  # Menggunakan teks asli sebagai referensi

# Menghitung BERTScore untuk setiap pasangan QA
print("BERT - BERTScore per QA Pair:\n")

for i, qa in enumerate(qa_list_PegasusXindo):
    candidate = qa['question'] + " " + qa['answer']
    P, R, F1 = scorer.score([candidate], [reference])

    print(f"QA {i+1}:")
    print(f"  Precision: {P.mean():.4f}")
    print(f"  Recall: {R.mean():.4f}")
    print(f"  F1-score: {F1.mean():.4f}\n")


BERT - BERTScore per QA Pair:

QA 1:
  Precision: 0.7927
  Recall: 0.6802
  F1-score: 0.7322

QA 2:
  Precision: 0.7608
  Recall: 0.6681
  F1-score: 0.7115

QA 3:
  Precision: 0.8108
  Recall: 0.6737
  F1-score: 0.7359

QA 4:
  Precision: 0.7999
  Recall: 0.6458
  F1-score: 0.7147

QA 5:
  Precision: 0.7762
  Recall: 0.6611
  F1-score: 0.7141

QA 6:
  Precision: 0.7928
  Recall: 0.6892
  F1-score: 0.7374

QA 7:
  Precision: 0.7431
  Recall: 0.6291
  F1-score: 0.6814

QA 8:
  Precision: 0.7827
  Recall: 0.6622
  F1-score: 0.7175



# Bobot

In [48]:
import numpy as np
from collections import defaultdict

# Fungsi untuk menampilkan rata-rata BERTScore untuk setiap model
def display_average_bert_score(qa_list, scorer, reference, model_name):
    f1_scores = []

    # Menghitung BERTScore untuk setiap QA pair
    for qa in qa_list:
        candidate = qa['question'] + " " + qa['answer']
        P, R, F1 = scorer.score([candidate], [reference])  # Menggunakan scorer BERTScore
        f1_scores.append(F1.mean().item())  # Ambil nilai F1-score
    
    # Menampilkan rata-rata BERTScore untuk model
    avg_f1 = np.mean(f1_scores)
    print(f"Average BERTScore for {model_name}: {avg_f1}")
    return avg_f1

In [49]:
# Asumsikan qa_list_bert, qa_list_t5, qa_list_pegasusx, scorer, reference sudah didefinisikan
bert_f1 = display_average_bert_score(qa_list_bert, scorer, reference, "BERT2BERT")
t5_f1 = display_average_bert_score(qa_list_t5, scorer, reference, "T5")
pegasus_f1 = display_average_bert_score(qa_list_pegasusx, scorer, reference, "PEGASUS-X")


Average BERTScore for BERT2BERT: 0.6773426383733749
Average BERTScore for T5: 0.6795627698302269
Average BERTScore for PEGASUS-X: 0.6780993491411209


In [50]:
# Menghitung total F1 score
total_f1 = bert_f1 + t5_f1 + pegasus_f1

In [51]:
# Pastikan F1 score tidak kosong atau NaN
if total_f1 > 0:
    model_weights = {
        "BERT": bert_f1 / total_f1,
        "T5": t5_f1 / total_f1,
        "PEGASUS": pegasus_f1 / total_f1
    }
else:
    # Jika total_f1 adalah 0 atau negatif, kita atur bobot menjadi sama
    model_weights = {
        "BERT": 1/3,
        "T5": 1/3,
        "PEGASUS": 1/3
    }

In [52]:
# Menampilkan bobot model
print(model_weights)

{'BERT': 0.3328457272292437, 'T5': 0.33393669836768414, 'PEGASUS': 0.33321757440307215}


# Probabilitas

## Token Probability

In [53]:
import torch
import torch.nn.functional as F
from typing import List, Mapping
import re

def show_qa_tokens_with_probabilities(qa_list: List[Mapping[str, str]], tokenizer, model, model_name: str = "") -> None:
    print(f"\n{'='*30}\nTokens and Probabilities from {model_name}\n{'='*30}")
    
    for i, qa in enumerate(qa_list):
        question_raw = qa['question']
        answer_raw = qa['answer']

        # Process question tokens
        q_tokens = tokenizer.tokenize(question_raw)
        q_tokens_cleaned = [re.sub(r'[^a-zA-Z0-9]+', '', token) for token in q_tokens]  # Remove non-alphanumeric characters

        print(f"\n{i+1}) Q: {question_raw}")  # Display the original question
        print("   Question Tokens with Probabilities:")
        inputs = tokenizer(question_raw, return_tensors="pt")
        input_ids = inputs["input_ids"].to(model.device)

        with torch.no_grad():
            decoder_input_ids = model.prepare_decoder_input_ids_from_labels(input_ids)
            outputs = model(input_ids=input_ids, decoder_input_ids=decoder_input_ids)
            logits = outputs.logits
            probabilities = F.softmax(logits, dim=-1)

        for idx, token in enumerate(q_tokens_cleaned):
            token_id = tokenizer.convert_tokens_to_ids([q_tokens[idx]])[0]
            if token_id < probabilities.shape[-1]:
                prob = probabilities[0, idx, token_id].item()
                print(f"      {token}: {prob:.10f}")

        # Process answer tokens
        if isinstance(answer_raw, list):  # Multiple choice format
            print("   Answer Tokens with Probabilities (Multiple Choice):")
            for idx, ans in enumerate(answer_raw):
                ans_text_raw = ans['answer']
                ans_tokens = tokenizer.tokenize(ans_text_raw)
                ans_tokens_cleaned = [re.sub(r'[^a-zA-Z0-9]+', '', token) for token in ans_tokens]  # Clean tokens

                correct_tag = "(correct)" if ans['correct'] else ""
                print(f"      Choice {idx+1}: {ans_text_raw} {correct_tag}")

                inputs = tokenizer(ans_text_raw, return_tensors="pt")
                input_ids = inputs["input_ids"].to(model.device)

                with torch.no_grad():
                    decoder_input_ids = model.prepare_decoder_input_ids_from_labels(input_ids)
                    outputs = model(input_ids=input_ids, decoder_input_ids=decoder_input_ids)
                    logits = outputs.logits
                    probabilities = F.softmax(logits, dim=-1)

                for token in ans_tokens_cleaned:
                    token_id = tokenizer.convert_tokens_to_ids([token])[0]
                    if token_id < probabilities.shape[-1]:
                        prob = probabilities[0, idx, token_id].item()
                        print(f"         {token}: {prob:.10f}")
        else:
            print(f"   Answer: {answer_raw}")
            a_tokens = tokenizer.tokenize(answer_raw)
            a_tokens_cleaned = [re.sub(r'[^a-zA-Z0-9]+', '', token) for token in a_tokens]  # Clean tokens
            print("   Answer Tokens with Probabilities:")
            inputs = tokenizer(answer_raw, return_tensors="pt")
            input_ids = inputs["input_ids"].to(model.device)

            with torch.no_grad():
                decoder_input_ids = model.prepare_decoder_input_ids_from_labels(input_ids)
                outputs = model(input_ids=input_ids, decoder_input_ids=decoder_input_ids)
                logits = outputs.logits
                probabilities = F.softmax(logits, dim=-1)

            for token in a_tokens_cleaned:
                token_id = tokenizer.convert_tokens_to_ids([token])[0]
                if token_id < probabilities.shape[-1]:
                    prob = probabilities[0, idx, token_id].item()
                    print(f"      {token}: {prob:.10f}")



In [54]:
show_qa_tokens_with_probabilities(qa_list_bert, tokenizer_bert, model_bert, "BERT")


Tokens and Probabilities from BERT

1) Q: What is the national symbol of Monas?
   Question Tokens with Probabilities:
      What: 0.0155844791
      is: 0.0000004406
      the: 0.0000022305
      national: 0.0000003999
      symbol: 0.0001042723
      of: 0.0023994714
      Mona: 0.0018701490
      s: 0.0000000131
      : 0.0000000008
   Answer: Monas or National Monument is an iconic landmark located in Jakarta, Indonesia.
   Answer Tokens with Probabilities:
      Mona: 0.0104105081
      s: 0.0000000067
      or: 0.0000044722
      National: 0.0000001022
      Monument: 0.0000037077
      is: 0.0000685987
      an: 0.0000115490
      i: 0.0000114775
      con: 0.0000000037
      ic: 0.0000062255
      landmark: 0.0000009114
      located: 0.0000151659
      in: 0.0000018725
      Jakarta: 0.0000000147
      : 0.0000062255
      Indonesia: 0.0000662956
      : 0.0000062255

2) Q: What does the Monas symbolize independence?
   Question Tokens with Probabilities:
      What: 0.000113

In [55]:
show_qa_tokens_with_probabilities(qa_list_t5, tokenizer_t5, model_t5, "T5")


Tokens and Probabilities from T5

1) Q: What is the name of the iconic landmark in Jakarta?
   Question Tokens with Probabilities:
      What: 0.9060162902
      is: 0.7712897062
      the: 0.9680736661
      name: 0.9051262736
      of: 0.9934717417
      the: 0.8876519203
      iconic: 0.5994294286
      landmark: 0.9319003224
      in: 0.9563714266
      Jakarta: 0.9893193841
      : 0.9868887663
   Answer: Monas is not only a symbol of national struggle, but also an important part of Indonesian cultural identity.
   Answer Tokens with Probabilities:
      Mon: 0.0000000006
      a: 0.0000012614
      s: 0.0000040908
      is: 0.0000009711
      not: 0.0000000168
      only: 0.0000000006
      : 0.0000006293
      a: 0.0000012614
      symbol: 0.0000006293
      of: 0.0000000271
      national: 0.0000359017
      struggle: 0.0000006293
      : 0.0000006293
      but: 0.0000000372
      also: 0.0000000016
      an: 0.0000003275
      important: 0.0000000003
      part: 0.0000000015


In [56]:
show_qa_tokens_with_probabilities(qa_list_pegasusx, tokenizer_pegasusx, model_pegasusx, "PEGASUS-X")


Tokens and Probabilities from PEGASUS-X

1) Q: How much do you know about the Monas monument in Jakarta, Indonesia?
   Question Tokens with Probabilities:
      How: 0.0388677083
      much: 0.7964359522
      do: 0.8382796049
      you: 0.9774187207
      know: 0.7831858993
      about: 0.9104790092
      the: 0.7863260508
      Mona: 0.8979536295
      s: 0.9371974468
      monument: 0.8585194349
      in: 0.8804587722
      Jakarta: 0.7972249389
      : 0.8897927403
      Indonesia: 0.8093741536
      : 0.8973964453
   Answer: Monas is one of the main tourist destinations in the Capital City and is often used as a place for state ceremonies and various other major events.
   Answer Tokens with Probabilities:
      Mona: 0.0000030997
      s: 0.0000017477
      is: 0.0000868091
      one: 0.0000008535
      of: 0.0000002742
      the: 0.0000021920
      main: 0.0000015525
      tourist: 0.0000026857
      destinations: 0.0000147384
      in: 0.0000029228
      the: 0.0000021920
    

## Threshold

In [57]:
import torch
import torch.nn.functional as F
import numpy as np

def compute_global_threshold(qa_list_all_models: List[tuple]):
    all_probs = []

    # Collect all token probabilities from all models
    for qa_list, tokenizer, model, _ in qa_list_all_models:
        for qa in qa_list:
            for text in [qa['question'], qa['answer'] if isinstance(qa['answer'], str) else None]:
                if not text:
                    continue
                tokens = tokenizer.tokenize(text)
                tokens_cleaned = [token.lstrip('_') for token in tokens]
                inputs = tokenizer(text, return_tensors="pt")
                input_ids = inputs["input_ids"].to(model.device)
                with torch.no_grad():
                    decoder_input_ids = model.prepare_decoder_input_ids_from_labels(input_ids)
                    outputs = model(input_ids=input_ids, decoder_input_ids=decoder_input_ids)
                    logits = outputs.logits
                    probabilities = F.softmax(logits, dim=-1)

                for idx, token in enumerate(tokens_cleaned):
                    token_id = tokenizer.convert_tokens_to_ids([token])[0]
                    if idx < probabilities.shape[1] and token_id < probabilities.shape[2]:
                        prob = probabilities[0, idx, token_id].item()
                        all_probs.append(prob)

    # Calculate the median threshold
    threshold = np.median(all_probs)
    return threshold


In [58]:
# Example of calling the function and displaying the threshold
threshold = compute_global_threshold([
    (qa_list_bert, tokenizer_bert, model_bert, "BERT"),
    (qa_list_t5, tokenizer_t5, model_t5, "T5"),
    (qa_list_pegasusx, tokenizer_pegasusx, model_pegasusx, "PEGASUS-X")
])

print(f"Global Threshold (Median Probability): {threshold:.10f}")

Global Threshold (Median Probability): 0.7107917666


## Token Probability with Threshold

In [59]:
import torch
import torch.nn.functional as F

def display_tokens_above_threshold_for_model(qa_list, tokenizer, model, model_name, threshold):
    # Step: Re-process and display only tokens above the threshold for the current model
    print(f"\n{'='*20} TOKENS ABOVE GLOBAL THRESHOLD: {model_name} {'='*20}")
    
    for i, qa in enumerate(qa_list):
        print(f"\n{i+1}) Q: {qa['question']}")
        
        # Process question tokens
        q_tokens = tokenizer.tokenize(qa['question'])
        q_tokens_cleaned = [token.lstrip('_').rstrip('_') for token in q_tokens]  # Clean leading and trailing underscores
        inputs = tokenizer(qa['question'], return_tensors="pt")
        input_ids = inputs["input_ids"].to(model.device)
        
        with torch.no_grad():
            decoder_input_ids = model.prepare_decoder_input_ids_from_labels(input_ids)
            outputs = model(input_ids=input_ids, decoder_input_ids=decoder_input_ids)
            probs = F.softmax(outputs.logits, dim=-1)  # Get probabilities from logits

        print("   Question Tokens Above Threshold:")
        for idx, token in enumerate(q_tokens_cleaned):
            token_id = tokenizer.convert_tokens_to_ids([q_tokens[idx]])[0]
            if idx < probs.shape[1] and token_id < probs.shape[2]:
                prob = probs[0, idx, token_id].item()
                if prob >= threshold:
                    print(f"      {token}: {prob:.10f}")

        # Process answer tokens
        if isinstance(qa['answer'], str):
            print(f"   A: {qa['answer']}")
            a_tokens = tokenizer.tokenize(qa['answer'])
            a_tokens_cleaned = [token.lstrip('_').rstrip('_') for token in a_tokens]  # Clean answer tokens
            inputs = tokenizer(qa['answer'], return_tensors="pt")
            input_ids = inputs["input_ids"].to(model.device)
            
            with torch.no_grad():
                decoder_input_ids = model.prepare_decoder_input_ids_from_labels(input_ids)
                outputs = model(input_ids=input_ids, decoder_input_ids=decoder_input_ids)
                probs = F.softmax(outputs.logits, dim=-1)  # Get probabilities from logits

            print("   Answer Tokens Above Threshold:")
            for idx, token in enumerate(a_tokens_cleaned):
                token_id = tokenizer.convert_tokens_to_ids([token])[0]
                if idx < probs.shape[1] and token_id < probs.shape[2]:
                    prob = probs[0, idx, token_id].item()
                    if prob >= threshold:
                        print(f"      {token}: {prob:.10f}")

In [60]:
# Example of calling the functions to display each model's output separately

# For BERT
display_tokens_above_threshold_for_model(qa_list_bert, tokenizer_bert, model_bert, "BERT", threshold)


==================== TOKENS ABOVE GLOBAL THRESHOLD: BERT ====================

1) Q: What is the national symbol of Monas?
   Question Tokens Above Threshold:
   A: Monas or National Monument is an iconic landmark located in Jakarta, Indonesia.
   Answer Tokens Above Threshold:

2) Q: What does the Monas symbolize independence?
   Question Tokens Above Threshold:
   A: Monas is not only a symbol of national struggle, but also an important part of Indonesian cultural identity.
   Answer Tokens Above Threshold:

3) Q: What does Monas symbolize?
   Question Tokens Above Threshold:
   A: In addition, the area around Monas is also a green open space that is often used by the community for sports and recreation.
   Answer Tokens Above Threshold:
      used: 0.7522000074

4) Q: What is the origin of the name Monas?
   Question Tokens Above Threshold:
   A: Monas is one of the main tourist destinations in the Capital City and is often used as a place for state ceremonies and various other maj

In [61]:
# For T5
display_tokens_above_threshold_for_model(qa_list_t5, tokenizer_t5, model_t5, "T5", threshold)



==================== TOKENS ABOVE GLOBAL THRESHOLD: T5 ====================

1) Q: What is the name of the iconic landmark in Jakarta?
   Question Tokens Above Threshold:
      ▁What: 0.9060162902
      ▁is: 0.7712897062
      ▁the: 0.9680736661
      ▁name: 0.9051262736
      ▁of: 0.9934717417
      ▁the: 0.8876519203
      ▁landmark: 0.9319003224
      ▁in: 0.9563714266
      ▁Jakarta: 0.9893193841
      ?: 0.9868887663
   A: Monas is not only a symbol of national struggle, but also an important part of Indonesian cultural identity.
   Answer Tokens Above Threshold:
      a: 0.9953734279
      s: 0.9993995428
      ▁is: 0.8622019887
      ▁only: 0.8798051476
      ▁: 0.9196160436
      a: 0.9995658994
      ▁symbol: 0.9877405763
      ▁of: 0.9086924791
      ▁struggle: 0.9913431406
      ,: 0.9004350901
      ▁but: 0.8404250145
      ▁an: 0.7911763787
      ▁important: 0.9949992895
      ▁part: 0.9924116731
      ▁of: 0.9955275655
      n: 0.7148511410
      ▁identity: 0.9711389542


In [62]:

# For PEGASUS-X
display_tokens_above_threshold_for_model(qa_list_pegasusx, tokenizer_pegasusx, model_pegasusx, "PEGASUS-X", threshold)


==================== TOKENS ABOVE GLOBAL THRESHOLD: PEGASUS-X ====================

1) Q: How much do you know about the Monas monument in Jakarta, Indonesia?
   Question Tokens Above Threshold:
      ▁much: 0.7964359522
      ▁do: 0.8382796049
      ▁you: 0.9774187207
      ▁know: 0.7831858993
      ▁about: 0.9104790092
      ▁the: 0.7863260508
      ▁Mona: 0.8979536295
      s: 0.9371974468
      ▁monument: 0.8585194349
      ▁in: 0.8804587722
      ▁Jakarta: 0.7972249389
      ,: 0.8897927403
      ▁Indonesia: 0.8093741536
      ?: 0.8973964453
   A: Monas is one of the main tourist destinations in the Capital City and is often used as a place for state ceremonies and various other major events.
   Answer Tokens Above Threshold:
      s: 0.8903334737
      ▁of: 0.9201821685
      ▁the: 0.7325308919
      ▁tourist: 0.9011851549
      ▁destinations: 0.7642085552
      ▁in: 0.8781307936
      ▁the: 0.8493239880
      ▁City: 0.8260713220
      ▁and: 0.7784848809
      ▁often: 0.8704266

# Ensemble Learning

## Weighted Averaging

In [63]:
def extract_tokens_with_probs(text, tokenizer, model, threshold):
    import torch
    import torch.nn.functional as F

    tokens = tokenizer.tokenize(text)
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"].to(model.device)

    with torch.no_grad():
        decoder_input_ids = model.prepare_decoder_input_ids_from_labels(input_ids)
        outputs = model(input_ids=input_ids, decoder_input_ids=decoder_input_ids)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1)

    token_probs = {}
    for idx, token in enumerate(tokens):
        token_id = tokenizer.convert_tokens_to_ids([token])[0]
        if idx < probs.shape[1] and token_id < probs.shape[2]:
            prob = probs[0, idx, token_id].item()
            if prob >= threshold:
                token_probs[token] = prob

    return token_probs


In [64]:
def compute_weighted_token_scores(token_prob_dicts, model_weights):
    from collections import defaultdict

    token_score_sum = defaultdict(float)
    token_weight_sum = defaultdict(float)

    for model_name, token_probs in token_prob_dicts.items():
        weight = model_weights[model_name]
        for token, prob in token_probs.items():
            token_score_sum[token] += prob * weight
            token_weight_sum[token] += weight

    final_token_scores = {
        token: token_score_sum[token] / token_weight_sum[token]
        for token in token_score_sum
    }
    return final_token_scores


In [65]:
def compute_weighted_token_scores(token_prob_dicts, model_weights):
    from collections import defaultdict

    token_score_sum = defaultdict(float)
    token_weight_sum = defaultdict(float)

    for model_name, token_probs in token_prob_dicts.items():
        weight = model_weights[model_name]
        for token, prob in token_probs.items():
            token_score_sum[token] += prob * weight
            token_weight_sum[token] += weight

    final_token_scores = {
        token: token_score_sum[token] / token_weight_sum[token]
        for token in token_score_sum
    }
    return final_token_scores


In [66]:
def select_tokens_above_threshold(token_scores, threshold):
    return [token for token, score in token_scores.items() if score >= threshold]


In [67]:
def assemble_tokens_to_text(tokens, tokenizer):
    # Gabungkan token dan hilangkan "▁" di awal setiap token untuk memperbaiki hasil
    cleaned_tokens = [token.replace("▁", "") for token in tokens]
    if not cleaned_tokens:
        return "[NO TOKEN SELECTED]"
    return tokenizer.convert_tokens_to_string(cleaned_tokens)


In [68]:
def ensemble_single_qa_pair(
    qa_lists_by_model,
    model_weights,
    tokenizer_by_model,
    model_by_name,
    threshold
):
    token_prob_dicts_q = {}
    token_prob_dicts_a = {}

    # Proses semua QA pairs untuk question dan answer
    for model_name, qa_list in qa_lists_by_model.items():
        for qa in qa_list:
            question_text = qa['question']
            answer_text = qa['answer']
            
            # Ekstraksi token dan probabilitas untuk question
            tokenizer = tokenizer_by_model[model_name]
            model = model_by_name[model_name]
            token_probs_q = extract_tokens_with_probs(question_text, tokenizer, model, threshold)
            token_prob_dicts_q[model_name] = token_probs_q

            # Ekstraksi token dan probabilitas untuk answer
            token_probs_a = extract_tokens_with_probs(answer_text, tokenizer, model, threshold)
            token_prob_dicts_a[model_name] = token_probs_a

    # Gabungkan token dari seluruh model untuk question dan answer
    question_token_scores = compute_weighted_token_scores(token_prob_dicts_q, model_weights)
    selected_question_tokens = select_tokens_above_threshold(question_token_scores, threshold)
    final_question = assemble_tokens_to_text(selected_question_tokens, tokenizer_by_model["BERT"])

    answer_token_scores = compute_weighted_token_scores(token_prob_dicts_a, model_weights)
    selected_answer_tokens = select_tokens_above_threshold(answer_token_scores, threshold)
    final_answer = assemble_tokens_to_text(selected_answer_tokens, tokenizer_by_model["BERT"])

    return {
        "question": final_question,
        "answer": final_answer
    }


In [69]:
def ensemble_all_qa_pairs(
    qa_lists_by_model,
    model_weights,
    tokenizer_by_model,
    model_by_name,
    threshold
):
    # Asumsikan semua model menghasilkan jumlah QA yang sama
    total_qa = len(next(iter(qa_lists_by_model.values())))
    all_ensemble_qa = []

    for i in range(total_qa):
        final_qa = ensemble_qa_pair(
            qa_index=i,
            qa_lists_by_model=qa_lists_by_model,
            model_weights=model_weights,
            tokenizer_by_model=tokenizer_by_model,
            model_by_name=model_by_name,
            threshold=threshold
        )
        all_ensemble_qa.append(final_qa)

    return all_ensemble_qa


In [70]:
# Pastikan dictionary berikut sudah ada dari sebelumnya
qa_lists_by_model = {
    "BERT": qa_list_bert,
    "T5": qa_list_t5,
    "PEGASUS": qa_list_pegasusx
}

tokenizer_by_model = {
    "BERT": tokenizer_bert,
    "T5": tokenizer_t5,
    "PEGASUS": tokenizer_pegasusx
}

model_by_name = {
    "BERT": model_bert,
    "T5": model_t5,
    "PEGASUS": model_pegasusx
}

threshold = 0.7107917666  # Threshold global yang sudah dihitung sebelumnya

# Panggil ensemble untuk menghasilkan 1 QA pair final
final_qa = ensemble_single_qa_pair(
    qa_lists_by_model=qa_lists_by_model,
    model_weights=model_weights,
    tokenizer_by_model=tokenizer_by_model,
    model_by_name=model_by_name,
    threshold=threshold
)

# Cetak hasilnya
print("❓ Final Question:", final_qa["question"])
print("✅ Final Answer:", final_qa["answer"])


❓ Final Question: Warum was the National Monument built? much do you know about s of Indonesia
✅ Final Answer: independence area around Mon a s is green open space that used the community for and. go up to top of monument using an elevator enjoy view city Jakarta a height


## Voting

### Per Token

In [71]:
def extract_tokens_with_probs(text, tokenizer, model, threshold):
    import torch
    import torch.nn.functional as F

    tokens = tokenizer.tokenize(text)
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"].to(model.device)

    with torch.no_grad():
        decoder_input_ids = model.prepare_decoder_input_ids_from_labels(input_ids)
        outputs = model(input_ids=input_ids, decoder_input_ids=decoder_input_ids)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1)

    token_probs = {}
    for idx, token in enumerate(tokens):
        token_id = tokenizer.convert_tokens_to_ids([token])[0]
        if idx < probs.shape[1] and token_id < probs.shape[2]:
            prob = probs[0, idx, token_id].item()
            if prob >= threshold:
                token_probs[token] = prob

    return token_probs

In [72]:
def compute_token_scores(token_prob_dicts):
    """
    Calculates the average token scores across all models.
    """
    from collections import defaultdict

    token_score_sum = defaultdict(float)
    token_count = defaultdict(int)

    for model_name, token_probs in token_prob_dicts.items():
        for token, prob in token_probs.items():
            token_score_sum[token] += prob
            token_count[token] += 1

    final_token_scores = {
        token: token_score_sum[token] / token_count[token]
        for token in token_score_sum
    }
    return final_token_scores

In [73]:
def select_tokens_above_threshold(token_scores, threshold):
    return [token for token, score in token_scores.items() if score >= threshold]

In [74]:
def assemble_tokens_to_text(tokens, tokenizer):
    # Gabungkan token dan hilangkan "▁" di awal setiap token untuk memperbaiki hasil
    cleaned_tokens = [token.replace("▁", "") for token in tokens]
    if not cleaned_tokens:
        return "[NO TOKEN SELECTED]"
    return tokenizer.convert_tokens_to_string(cleaned_tokens)

In [75]:
def ensemble_single_qa_pair(
    qa_lists_by_model,
    tokenizer_by_model,
    model_by_name,
    threshold
):
    """
    Ensembles a single QA pair by averaging token probabilities
    across all models without weights.
    """
    token_prob_dicts_q = {}
    token_prob_dicts_a = {}

    # Process all QA pairs for question and answer
    for model_name, qa_list in qa_lists_by_model.items():
        for qa in qa_list:
            question_text = qa['question']
            answer_text = qa['answer']

            # Extract tokens and probabilities for question
            tokenizer = tokenizer_by_model[model_name]
            model = model_by_name[model_name]
            token_probs_q = extract_tokens_with_probs(question_text, tokenizer, model, threshold)
            token_prob_dicts_q[model_name] = token_probs_q

            # Extract tokens and probabilities for answer
            token_probs_a = extract_tokens_with_probs(answer_text, tokenizer, model, threshold)
            token_prob_dicts_a[model_name] = token_probs_a

    # Combine tokens from all models for question and answer
    question_token_scores = compute_token_scores(token_prob_dicts_q)  # Use unweighted average
    selected_question_tokens = select_tokens_above_threshold(question_token_scores, threshold)
    final_question = assemble_tokens_to_text(selected_question_tokens, tokenizer_by_model["BERT"])

    answer_token_scores = compute_token_scores(token_prob_dicts_a)  # Use unweighted average
    selected_answer_tokens = select_tokens_above_threshold(answer_token_scores, threshold)
    final_answer = assemble_tokens_to_text(selected_answer_tokens, tokenizer_by_model["BERT"])

    return {
        "question": final_question,
        "answer": final_answer
    }

In [76]:
def ensemble_all_qa_pairs(
    qa_lists_by_model,
    tokenizer_by_model,
    model_by_name,
    threshold
):
    """
    Ensembles all QA pairs using the unweighted averaging approach.
    """
    total_qa = len(next(iter(qa_lists_by_model.values())))  # Assumes all models have the same number of QAs
    all_ensemble_qa = []

    for i in range(total_qa):
        final_qa = ensemble_single_qa_pair(  # Calls the updated ensemble_single_qa_pair without model_weights
            qa_lists_by_model=qa_lists_by_model,
            tokenizer_by_model=tokenizer_by_model,
            model_by_name=model_by_name,
            threshold=threshold
        )
        all_ensemble_qa.append(final_qa)

    return all_ensemble_qa

In [77]:
# Pastikan dictionary berikut sudah ada dari sebelumnya
qa_lists_by_model = {
    "BERT": qa_list_bert,
    "T5": qa_list_t5,
    "PEGASUS": qa_list_pegasusx
}

tokenizer_by_model = {
    "BERT": tokenizer_bert,
    "T5": tokenizer_t5,
    "PEGASUS": tokenizer_pegasusx
}

model_by_name = {
    "BERT": model_bert,
    "T5": model_t5,
    "PEGASUS": model_pegasusx
}

# Panggil ensemble untuk menghasilkan 1 QA pair final
final_qa = ensemble_single_qa_pair(
    qa_lists_by_model=qa_lists_by_model,
    tokenizer_by_model=tokenizer_by_model,
    model_by_name=model_by_name,
    threshold=threshold
)

# Cetak hasilnya
print("❓ Final Question:", final_qa["question"])
print("✅ Final Answer:", final_qa["answer"])

❓ Final Question: Warum was the National Monument built? much do you know about s of Indonesia
✅ Final Answer: independence area around Mon a s is green open space that used the community for and. go up to top of monument using an elevator enjoy view city Jakarta a height


### Per QA List

In [78]:
def compute_bert_score_per_qa(translated_qa_list, model_name, print_scores=False):
    """
    Menghitung BERTScore untuk setiap QA pair dalam list model tertentu.
    Menyimpan hasilnya dalam dictionary dan hanya mencetak jika print_scores = True.
    """
    model_scores = []  # List untuk menyimpan hasil BERTScore dari model

    for i, qa_pair in enumerate(translated_qa_list, 1):
        question = qa_pair.get("question", "").strip()
        answer = qa_pair.get("answer", "").strip()
        if not question or not answer:
            continue  # Skip jika tidak ada pertanyaan atau jawaban

        qa_text = f"{question} {answer}"  # Gabungkan pertanyaan dan jawaban

        # Hitung BERTScore untuk QA pair ini
        P, R, F1 = scorer.score([qa_text], [text_context])

        # Simpan hasil dalam dictionary
        bert_score_result = {
            "QA_index": i,
            "Precision": P.mean().item(),
            "Recall": R.mean().item(),
            "F1": F1.mean().item()
        }
        model_scores.append(bert_score_result)

        # Cetak hasil hanya jika print_scores True
        if print_scores:
            print(f"{model_name} - QA {i}:")
            print(f"  Precision: {bert_score_result['Precision']:.4f}")
            print(f"  Recall: {bert_score_result['Recall']:.4f}")
            print(f"  F1-score: {bert_score_result['F1']:.4f}\n")

    return model_scores  # Mengembalikan daftar hasil skor BERTScore

# Hitung skor BERTScore untuk setiap QA pair dari masing-masing model dan simpan hasilnya
# Jika Anda sudah menampilkan skor sebelumnya, set print_scores=False
t5_scores = compute_bert_score_per_qa(qa_list_t5, "T5", print_scores=False)
pegasusx_scores = compute_bert_score_per_qa(qa_list_pegasusx, "PEGASUS-X", print_scores=False)
bert_scores = compute_bert_score_per_qa(qa_list_bert, "BERT", print_scores=False)

# Simpan semua hasil dalam satu dictionary untuk memudahkan voting
all_model_scores = {
    "T5": t5_scores,
    "PEGASUS-X": pegasusx_scores,
    "BERT": bert_scores
}


In [79]:
# Dictionary yang menyimpan QA Pairs dari setiap model
all_model_qa_pairs = {
    "T5": qa_list_t5,  # List QA Pairs dari T5
    "PEGASUS-X": qa_list_pegasusx,  # List QA Pairs dari PEGASUS-X
    "BERT": qa_list_bert  # List QA Pairs dari BERT
}

# Hitung rata-rata skor tiap model untuk menentukan bobot
model_avg_scores = {}
for model, scores in all_model_scores.items():
    if scores:  # Pastikan model memiliki QA Pairs
        avg_precision = np.mean([s["Precision"] for s in scores])
        avg_recall = np.mean([s["Recall"] for s in scores])
        avg_f1 = np.mean([s["F1"] for s in scores])

        # Mean Score untuk setiap model
        model_avg_scores[model] = np.mean([avg_precision, avg_recall, avg_f1])

# Normalisasi bobot model berdasarkan rata-rata skor
total_avg_score = sum(model_avg_scores.values())
weights = {model: model_avg_scores[model] / total_avg_score for model in model_avg_scores}

# Hitung Mean Score dan Weighted Voting untuk setiap QA Pair
qa_weighted_scores = []
qa_texts = {}  # Dictionary untuk menyimpan teks QA Pair terbaik
qa_model_source = {}  # Dictionary untuk menyimpan dari model mana QA Pair berasal

max_qa_pairs = max(len(v) for v in all_model_scores.values())  # Cari jumlah QA pairs terbanyak

for i in range(max_qa_pairs):
    weighted_score = 0
    count_models = 0  # Hitung model yang memiliki QA Pair ke-i
    best_qa_text = None  # Simpan teks QA Pair terbaik
    best_model = None  # Simpan model yang menghasilkan QA Pair terbaik

    for model, scores in all_model_scores.items():
        if i < len(scores):  # Pastikan model memiliki QA Pair ke-i
            mean_score = np.mean([scores[i]["Precision"], scores[i]["Recall"], scores[i]["F1"]])
            weighted_score += mean_score * weights[model]
            count_models += 1

            # Simpan QA Pair dan modelnya
            if model in all_model_qa_pairs and i < len(all_model_qa_pairs[model]):
                best_qa_text = all_model_qa_pairs[model][i]
                best_model = model  # Simpan nama model

    if count_models > 0:
        weighted_score /= count_models  # Rata-rata jika ada model yang tidak memiliki QA Pair ke-i
        qa_weighted_scores.append((i + 1, weighted_score))
        qa_texts[i + 1] = best_qa_text  # Simpan QA Pair terkait
        qa_model_source[i + 1] = best_model  # Simpan model yang menghasilkan QA Pair

# Pilih QA Pair dengan skor tertinggi
best_qa_pair_index, best_score = max(qa_weighted_scores, key=lambda x: x[1])
best_qa_pair = qa_texts.get(best_qa_pair_index, {"question": "N/A", "answer": "N/A"})
best_model_source = qa_model_source.get(best_qa_pair_index, "Unknown")

# Cetak hasil akhir
print(f"\nQA Pair terbaik adalah indeks ke-{best_qa_pair_index} dengan skor tertinggi: {best_score:.4f}")
print(f"🔹 Model yang Menghasilkan QA Pair Terbaik**: {best_model_source}")
print(f"🔹 Pertanyaan Terbaik**: {best_qa_pair['question']}")
print(f"🔹 Jawaban Terbaik**: {best_qa_pair['answer']}")


QA Pair terbaik adalah indeks ke-6 dengan skor tertinggi: 0.2331
🔹 Model yang Menghasilkan QA Pair Terbaik**: BERT
🔹 Pertanyaan Terbaik**: What is the national symbol of Monas?
🔹 Jawaban Terbaik**: In Monas, there is a historical museum containing the diorama of the journey of the struggle of the Indonesian people.


## Attention Based

In [80]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

# Load pre-trained sentence embedding model
encoder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

def encode_qa_pairs(qa_list):
    questions = [qa["question"] for qa in qa_list]
    answers = [qa["answer"] for qa in qa_list]
    question_embeddings = encoder.encode(questions, convert_to_tensor=True, normalize_embeddings=True)
    answer_embeddings = encoder.encode(answers, convert_to_tensor=True, normalize_embeddings=True)
    return question_embeddings, answer_embeddings

# Encode QA pairs
question_t5, answer_t5 = encode_qa_pairs(qa_list_t5)
question_pegasusx, answer_pegasusx = encode_qa_pairs(qa_list_pegasusx)
question_bert, answer_bert = encode_qa_pairs(qa_list_bert)

def combine_question_answer(question_embeddings, answer_embeddings):
    return torch.cat([question_embeddings, answer_embeddings], dim=-1)

combined_t5 = combine_question_answer(question_t5, answer_t5)
combined_pegasusx = combine_question_answer(question_pegasusx, answer_pegasusx)
combined_bert = combine_question_answer(question_bert, answer_bert)

# Stack embeddings
combined_outputs = torch.stack([combined_t5, combined_pegasusx, combined_bert]).contiguous()
combined_outputs = combined_outputs.to('cpu')

# Apply Layer Normalization after combining embeddings
layer_norm = nn.LayerNorm(combined_outputs.shape[-1])
normalized_outputs = layer_norm(combined_outputs)

# Self-Attention with Residual Connection
class SelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, device='cpu')
        self.layer_norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x.permute(1, 0, 2)  # Ensure correct shape
        attn_output, _ = self.attention(x, x, x)
        return self.layer_norm(attn_output + x).permute(1, 0, 2)  # Residual Connection

self_attention = SelfAttention(embed_dim=normalized_outputs.shape[-1])
self_attended_outputs = self_attention(normalized_outputs)

# Optimized Cross-Attention
class CrossAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, device='cpu')
        self.layer_norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        q, k, v = x[0], x[1], x[2]
        attn_output, _ = self.attention(q, k, v)
        return self.layer_norm(attn_output + q)  # Residual Connection

cross_attention = CrossAttention(embed_dim=self_attended_outputs.shape[-1])
cross_attended_outputs = cross_attention(self_attended_outputs)

# MLP Decoder with Dropout
class MLPDecoder(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.fc1 = nn.Linear(embed_dim, embed_dim // 2)
        self.dropout = nn.Dropout(p=0.1)
        self.fc2 = nn.Linear(embed_dim // 2, embed_dim)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

mlp_decoder = MLPDecoder(embed_dim=cross_attended_outputs.shape[-1])
decoded_outputs = mlp_decoder(torch.mean(cross_attended_outputs, dim=0, keepdim=True))

embedding_dim = decoded_outputs.shape[-1] // 2
final_question_embeddings = decoded_outputs[:, :embedding_dim]
final_answer_embeddings = decoded_outputs[:, embedding_dim:]

print("Final Question Embeddings Shape:", final_question_embeddings.shape)
print("Final Answer Embeddings Shape:", final_answer_embeddings.shape)

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

c:\Users\USER\anaconda3\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Final Question Embeddings Shape: torch.Size([1, 768])
Final Answer Embeddings Shape: torch.Size([1, 768])


In [81]:
# Softmax Scoring with Temperature
tau = 0.5

# Pastikan decoded_outputs memiliki dimensi yang benar
if decoded_outputs.dim() == 1:
    decoded_outputs = decoded_outputs.unsqueeze(0)  # Tambahkan dimensi batch jika perlu

linear_projection = nn.Linear(decoded_outputs.shape[-1], 1, bias=False).to(decoded_outputs.device)
projected_scores = linear_projection(decoded_outputs).squeeze(-1)

# Pastikan projected_scores memiliki lebih dari satu elemen sebelum softmax
if projected_scores.shape[0] == 1:
    softmax_scores = torch.tensor([1.0], device=projected_scores.device)
else:
    softmax_scores = F.softmax(projected_scores / tau, dim=-1)

top_1_index = torch.argmax(softmax_scores).item()

# Identify the model source
qa_sources = [qa_list_t5, qa_list_pegasusx, qa_list_bert]
qa_lengths = [len(q) for q in qa_sources]

# Cek apakah ada QA list yang kosong
if any(length == 0 for length in qa_lengths):
    raise ValueError("One of the QA sources has no data!")

# Cumulative lengths untuk indexing
cumulative_lengths = [0] + [sum(qa_lengths[:i+1]) for i in range(len(qa_lengths))]

# Pastikan index valid
if top_1_index >= sum(qa_lengths):
    print("Error: top_1_index out of range")
else:
    selected_qa_list, selected_index = None, None
    for model_idx, cum_len in enumerate(cumulative_lengths[:-1]):
        if cum_len <= top_1_index < cumulative_lengths[model_idx + 1]:
            selected_qa_list = qa_sources[model_idx]
            selected_index = top_1_index - cum_len
            break

    if selected_qa_list is not None and selected_index is not None:
        print("Question:", selected_qa_list[selected_index]["question"])
        print("Answer:", selected_qa_list[selected_index]["answer"])
    else:
        print("Error: No valid QA pair found!")

Question: What is the name of the iconic landmark in Jakarta?
Answer: Monas is not only a symbol of national struggle, but also an important part of Indonesian cultural identity.
